In [0]:
%sql
-- =====================================
-- GRANTS
-- =====================================
-- Nota:
-- Debido a fallas recientes en la infraestructura de Azure Databricks,
-- se han presentado reinicios de sesión frecuentes (~15 minutos),
-- lo cual ha afectado la estabilidad del entorno durante el desarrollo.
--
-- Por esta razón, el proyecto fue migrado a Databricks Free Edition.
--
-- Los siguientes comandos representan la configuración esperada
-- en un entorno enterprise con Unity Catalog completo.
-- Sin embargo, en Databricks Free Edition no se soporta
-- la gestión granular de permisos (GRANTS), por lo que estos
-- no son funcionales en este entorno.
--
-- Se incluyen únicamente como referencia de buenas prácticas
-- para escenarios productivos.

In [0]:
%sql
-- ================================
-- 1. METASTORE PERMISSIONS
-- ================================

-- Permite al usuario crear external locations dentro del metastore
-- (necesario para conectar el Data Lake con Unity Catalog)
GRANT CREATE EXTERNAL LOCATION ON METASTORE TO `gerodmx@outlook.com`;

-- Permite crear catálogos (útil para estructura medallion: bronze/silver/gold)
GRANT CREATE CATALOG ON METASTORE TO `gerodmx@outlook.com`;

-- Permite crear storage credentials (opcional, pero recomendable)
GRANT CREATE STORAGE CREDENTIAL ON METASTORE TO `gerodmx@outlook.com`;
-- ================================
-- VALIDACIÓN DE PERMISOS
-- ================================

-- Verifica permisos aplicados en el Metastore
SHOW GRANTS ON METASTORE;

-- Verifica permisos aplicados en el Storage Credential
SHOW GRANTS ON STORAGE CREDENTIAL credential_bike;

In [0]:
%sql

-- =====================================
-- 2. VALIDACIÓN DE USUARIO
-- =====================================

-- Verifica si el usuario actual es metastore admin
SELECT current_user() AS current_user;

-- Verifica permisos aplicados en el metastore
SHOW GRANTS ON METASTORE;

In [0]:
%sql

-- =========================================
-- 3. STORAGE CREDENTIAL PERMISSIONS
-- =========================================

-- Este bloque otorga permisos sobre el STORAGE CREDENTIAL
-- necesario para poder acceder al Data Lake desde Unity Catalog

-- -----------------------------------------
-- Permiso: READ FILES
-- Permite leer archivos desde el storage (capa RAW, por ejemplo)
-- -----------------------------------------
GRANT READ FILES 
ON STORAGE CREDENTIAL credential_bike 
TO `gerodmx@outlook.com`;

-- -----------------------------------------
-- Permiso: WRITE FILES
-- Permite escribir archivos en el storage
-- (necesario para procesos ETL / silver / gold)
-- -----------------------------------------
GRANT WRITE FILES 
ON STORAGE CREDENTIAL credential_bike 
TO `gerodmx@outlook.com`;

-- -----------------------------------------
-- Permiso: CREATE EXTERNAL LOCATION
-- Permite crear external locations usando este credential
-- (CRÍTICO para arquitectura medallion)
-- -----------------------------------------
GRANT CREATE EXTERNAL LOCATION 
ON STORAGE CREDENTIAL credential_bike 
TO `gerodmx@outlook.com`;

In [0]:
%sql
-- =====================================
-- 3. STORAGE CREDENTIAL PERMISSIONS
-- =====================================

-- Este bloque otorga permisos sobre el STORAGE CREDENTIAL
-- necesario para poder acceder al Data Lake desde Unity Catalog

-- -------------------------------------
-- Permiso: READ FILES
-- Permite leer archivos desde el storage (capa RAW, por ejemplo)
-- -------------------------------------
GRANT READ FILES
ON STORAGE CREDENTIAL credential_bike
TO `data_engineers`;

-- -------------------------------------
-- Permiso: WRITE FILES
-- Permite escribir archivos en el storage
-- (necesario para procesos ETL / silver / gold)
-- -------------------------------------
GRANT WRITE FILES
ON STORAGE CREDENTIAL credential_bike
TO `data_engineers`;

-- -------------------------------------
-- Permiso: CREATE EXTERNAL LOCATION
-- Permite crear external locations usando este credential
-- (crítico para arquitectura medallion)
-- -------------------------------------
GRANT CREATE EXTERNAL LOCATION
ON STORAGE CREDENTIAL credential_bike
TO `admins`;



-- =====================================
-- 4. CATALOG PERMISSIONS
-- =====================================

-- Permisos sobre el catálogo completo
GRANT USAGE ON CATALOG catalog_ecobici TO `analysts`;
GRANT USAGE ON CATALOG catalog_ecobici TO `data_engineers`;

GRANT CREATE ON CATALOG catalog_ecobici TO `admins`;



-- =====================================
-- 5. SCHEMA PERMISSIONS
-- =====================================

-- Permisos por esquema (arquitectura medallion)

-- Lectura para analistas
GRANT USAGE ON SCHEMA catalog_ecobici.bronze TO `analysts`;
GRANT USAGE ON SCHEMA catalog_ecobici.silver TO `analysts`;
GRANT USAGE ON SCHEMA catalog_ecobici.golden TO `analysts`;

-- Escritura para ingeniería
GRANT CREATE ON SCHEMA catalog_ecobici.bronze TO `data_engineers`;
GRANT CREATE ON SCHEMA catalog_ecobici.silver TO `data_engineers`;
GRANT CREATE ON SCHEMA catalog_ecobici.golden TO `data_engineers`;



-- =====================================
-- 6. TABLE PERMISSIONS
-- =====================================

-- Acceso de solo lectura para analistas
GRANT SELECT ON TABLE catalog_ecobici.golden.ecobici_vs_lluvia TO `analysts`;
GRANT SELECT ON TABLE catalog_ecobici.golden.ecobici_vs_temperatura TO `analysts`;

-- Acceso completo para ingeniería
GRANT SELECT, INSERT, UPDATE, DELETE
ON TABLE catalog_ecobici.silver.ecobici
TO `data_engineers`;

GRANT SELECT, INSERT, UPDATE, DELETE
ON TABLE catalog_ecobici.silver.weather
TO `data_engineers`;



-- =====================================
-- 7. VOLUME PERMISSIONS
-- =====================================

-- Acceso a archivos en RAW

-- Lectura para analistas
GRANT READ VOLUME
ON VOLUME catalog_ecobici.raw.raw_files
TO `analysts`;

-- Lectura y escritura para ingeniería
GRANT READ VOLUME, WRITE VOLUME
ON VOLUME catalog_ecobici.raw.raw_files
TO `data_engineers`;



-- =====================================
-- 8. REVOKE (EJEMPLO)
-- =====================================

-- Ejemplo de revocación de permisos
REVOKE WRITE VOLUME
ON VOLUME catalog_ecobici.raw.raw_files
FROM `analysts`;



-- =====================================
-- 9. VALIDACIÓN FINAL
-- =====================================

-- Verifica permisos aplicados en diferentes niveles
SHOW GRANTS ON CATALOG catalog_ecobici;
SHOW GRANTS ON SCHEMA catalog_ecobici.silver;
SHOW GRANTS ON TABLE catalog_ecobici.golden.ecobici_vs_lluvia;
SHOW GRANTS ON STORAGE CREDENTIAL credential_bike;